In [1]:
# ! pip install open3d laspy pyransac3d

In [2]:
import laspy
import open3d as o3d
import numpy as np
import pyransac3d as pyrsc

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [3]:
import os
os.getcwd()

'/home/cmgira/cnefs-research/PCSv1/benchmark'

In [4]:
file_path = "./segmented_lidar/Lab1total_labeled_weight.las" # <-- Update this

print("Loading LAS file...")
las_file = laspy.read(file_path)

# Keep 1 out of every 5 points. Adjust this based on your machine's RAM.
skip = 5 

points_array = np.column_stack((
    las_file.x[::skip], 
    las_file.y[::skip], 
    las_file.z[::skip]
))

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_array)

print(f"Successfully loaded {len(pcd.points)} points.")

Loading LAS file...
Successfully loaded 16435148 points.


In [ ]:
print("Running DBSCAN clustering...")

# Tweak these if you find 0 clusters!
EPSILON = 0.05  # Try 0.5, 5.0, or 50.0 if it fails
MIN_POINTS = 100 

labels = np.array(pcd.cluster_dbscan(eps=EPSILON, min_points=MIN_POINTS, print_progress=True))

max_label = labels.max()
print(f"Found {max_label + 1} distinct clusters (excluding noise).")

# --- Safety Check added here ---
if max_label < 0:
    print("\n⚠️ ALGORITHM FAILED TO FIND CLUSTERS ⚠️")
    print("All points were classified as noise. Your 'eps' value is likely too small for the units of your LiDAR file.")
    print("FIX: Increase the EPSILON variable at the top of this cell and run it again.")
else:
    # Isolate the largest cluster safely
    unique_labels, counts = np.unique(labels[labels >= 0], return_counts=True)
    largest_cluster_label = unique_labels[np.argmax(counts)]

    target_indices = np.where(labels == largest_cluster_label)[0]
    target_pcd = pcd.select_by_index(target_indices)
    target_points = np.asarray(target_pcd.points)

    print(f"Isolated the largest pipe cluster with {len(target_points)} points.")

In [ ]:
print("Fitting RANSAC Cylinder...\n")
cylinder = pyrsc.Cylinder()

# thresh is the distance tolerance for points to be considered part of the cylinder
center, direction, radius, inliers = cylinder.fit(target_points, thresh=0.01)

print(f"--- Pipe Measurements ---")
print(f"Center Coordinate (X,Y,Z): {np.round(center, 4)}")
print(f"Orientation Vector: {np.round(direction, 4)}")
print(f"Calculated Radius: {radius * 100:.2f} cm")
print(f"Diameter: {(radius * 2) * 100:.2f} cm")

In [ ]:
# Paint the background point cloud gray
pcd.paint_uniform_color([0.5, 0.5, 0.5])

# Paint our isolated pipe red
target_pcd.paint_uniform_color([1, 0, 0])

# Open the interactive 3D viewer (Use your mouse to rotate and zoom!)
o3d.visualization.draw_geometries([pcd, target_pcd])